In [1]:
# ...existing code...
import fitz  # PyMuPDF
import os
from io import BytesIO
from PIL import Image

def _compress_image_to_target(img, output_path, max_size_kb=500,
                              max_quality=85, min_quality=20,
                              progressive=True, optimize=True, step=5):
    """
    将 PIL.Image 保存为 jpeg，确保文件不超过 max_size_kb（尽力而为）。
    会先二分查找 quality，再在必要时按比例缩小图片并重试。
    """
    target_bytes = max_size_kb * 1024
    img = img.convert("RGB")

    # 尝试在当前尺寸上通过调整 quality 达到目标（使用二分搜索）
    def try_quality(q):
        buf = BytesIO()
        img.save(buf, "JPEG", quality=q, optimize=optimize, progressive=progressive)
        return buf.getvalue()

    lo, hi = min_quality, max_quality
    best_data = None

    while lo <= hi:
        mid = (lo + hi) // 2
        data = try_quality(mid)
        size = len(data)
        if size <= target_bytes:
            best_data = data
            lo = mid + 1  # try higher quality
        else:
            hi = mid - 1  # reduce quality

    if best_data:
        with open(output_path, "wb") as f:
            f.write(best_data)
        return os.path.getsize(output_path)

    # 如果单靠 quality 不能满足，则循环缩小图片并重试
    # 每次缩小比例为 0.9（10%），直到到达最小宽度或达到目标
    w, h = img.size
    min_width = 200
    cur_img = img
    while w > min_width:
        w = int(w * 0.9)
        h = int(h * 0.9)
        cur_img = cur_img.resize((w, h), Image.LANCZOS)

        # 重新二分查找 quality
        lo, hi = min_quality, max_quality
        best_data = None
        def try_quality_on(img_obj, q):
            buf = BytesIO()
            img_obj.save(buf, "JPEG", quality=q, optimize=optimize, progressive=progressive)
            return buf.getvalue()

        while lo <= hi:
            mid = (lo + hi) // 2
            data = try_quality_on(cur_img, mid)
            size = len(data)
            if size <= target_bytes:
                best_data = data
                lo = mid + 1
            else:
                hi = mid - 1

        if best_data:
            with open(output_path, "wb") as f:
                f.write(best_data)
            return os.path.getsize(output_path)

    # 最后一次尝试：用最低质量保存当前尺寸
    buf = BytesIO()
    cur_img.save(buf, "JPEG", quality=min_quality, optimize=optimize, progressive=progressive)
    with open(output_path, "wb") as f:
        f.write(buf.getvalue())
    return os.path.getsize(output_path)


def convert_pdf_and_images(input_folder, dpi=150, max_width=1200,
                           max_size_kb=500, overwrite=True):
    """
    将文件夹内的 PDF 转换为 JPG，并对已有的 JPG/PNG 也做压缩，
    目标是每个输出 jpg 尽量不超过 max_size_kb（KB），同时支持缩放到 max_width。
    """
    if not os.path.exists(input_folder):
        print(f"找不到文件夹: {input_folder}")
        return

    for file_name in os.listdir(input_folder):
        lower = file_name.lower()
        file_path = os.path.join(input_folder, file_name)

        # 处理 PDF -> JPG
        if lower.endswith(".pdf"):
            output_name = os.path.splitext(file_name)[0] + ".jpg"
            output_path = os.path.join(input_folder, output_name)
            if not overwrite and os.path.exists(output_path):
                print(f"跳过已存在文件: {output_name}")
                continue

            doc = fitz.open(file_path)
            try:
                page = doc[0]
                zoom = dpi / 72.0
                mat = fitz.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat, alpha=False)
                img_bytes = pix.tobytes("png")
                img = Image.open(BytesIO(img_bytes)).convert("RGB")

                # 缩放到 max_width（若需要）
                w, h = img.size
                if max_width and w > max_width:
                    new_h = int(max_width * h / w)
                    img = img.resize((max_width, new_h), Image.LANCZOS)

                size = _compress_image_to_target(img, output_path, max_size_kb=max_size_kb)
                print(f"PDF 转换并压缩: {file_name} -> {output_name} ({size} bytes)")
            finally:
                doc.close()

        # 处理已存在的图片文件（jpg/jpeg/png）
        elif lower.endswith((".jpg", ".jpeg", ".png")):
            output_name = os.path.splitext(file_name)[0] + ".jpg"
            output_path = os.path.join(input_folder, output_name)

            # 如果不是覆盖且输出已存在且大小已满足要求则跳过
            if not overwrite and os.path.exists(output_path):
                if os.path.getsize(output_path) <= max_size_kb * 1024:
                    print(f"跳过已满足大小的文件: {output_name}")
                    continue

            img = Image.open(file_path).convert("RGB")

            # 按 max_width 缩放（保持纵横比）
            w, h = img.size
            if max_width and w > max_width:
                new_h = int(max_width * h / w)
                img = img.resize((max_width, new_h), Image.LANCZOS)

            size = _compress_image_to_target(img, output_path, max_size_kb=max_size_kb)
            print(f"图片压缩: {file_name} -> {output_name} ({size} bytes)")

        else:
            # 非处理类型跳过
            continue


if __name__ == "__main__":
    target_dir = "certifications"
    convert_pdf_and_images(target_dir, dpi=150, max_width=1200, max_size_kb=500, overwrite=True)
# ...existing code...

In [2]:
import os

def generate_certifications_html(folder_path="certifications"):
    # 支持的图片格式
    extensions = (".jpg", ".jpeg", ".png", ".webp")
    
    # 获取文件夹内所有图片并排序（按文件名排序，建议文件名以年份开头，如 2012_xxx.jpg）
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(extensions)]
    files.sort(reverse=True)  # reverse=True 可以让最新的证书排在前面

    # 构建内部的 article 列表
    articles = []
    for file in files:
        alt_text = os.path.splitext(file)[0]
        
        article_template = f"""                                        <article>
                                            <a href="{folder_path}/{file}" class="image" target="_blank" style="border: 0;">
                                                <img src="{folder_path}/{file}" alt="{alt_text}" style="height: 400px; object-fit: contain; background: rgba(255,255,255,0.03); border-radius: 0px;" />
                                            </a>
                                        </article>"""
        articles.append(article_template)
    
    
    # 组合成完整的 section
    full_html = f"""
                            <section id="certifications" class="wrapper style2">
                                <div class="inner">
                                    <h2 class="major">Certifications</h2>
                                    <section class="features">
{''.join(articles)}
                                    </section>
                                </div>
                            </section>
    """
    
    return full_html

if __name__ == "__main__":
    result = generate_certifications_html()
    
    # 打印结果，你可以直接复制
    print(result)
    
    # 或者直接保存到一个文本文件里，方便查看
    with open("cert_output.txt", "w", encoding="utf-8") as f:
        f.write(result)
        print("\n--- HTML 代码已生成至 cert_output.txt ---")